In [ ]:
import picsure
import pandas as pd
from fastavro import reader

In [ ]:
token_file = "token.txt"

with open(token_file, "r") as f:
    my_token = f.read()

session = picsure.connect(
    platform=picsure.Platform.NHANES_AUTHORIZED,
    token=my_token,
    dev_mode=True,
)

In [ ]:
# Find the concepts we want to filter on and export.
facets = session.facets()
facets.add("dataset_id", "phs000007")

age_results = session.searchDictionary("age", facets=facets)
age5 = age_results[age_results["display"] == "age5"]
age5 = age5[age5["name"] == "phv00177938"]

sex_results = session.searchDictionary("phv00253990", facets=facets)

In [ ]:
# Filter to ages 30-40, then select age and sex as output columns.
age_filter = picsure.buildClause(
    age5["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40,
)

query = picsure.buildQuery(
    phenotypicFilter=age_filter,
    includeConcepts=[
        age5["conceptPath"].iloc[0],
        sex_results["conceptPath"].iloc[0],
    ],
)

In [ ]:
# Sanity-check the cohort size before exporting.
session.runQuery(query, type=picsure.QueryType.COUNT).value

In [ ]:
# exportAsPFB writes the includeConcepts columns for matching
# participants directly to disk as a PFB (Avro) file.
session.exportAsPFB(query=query, path="./cohort.avro")

In [ ]:
# Read the PFB back to confirm the selected columns are present.
def avro_to_df(path):
    with open(path, "rb") as f:
        records = list(reader(f))
    return pd.DataFrame(records)

df = avro_to_df("./cohort.avro")
df